In [1]:
"""
benchmark.py - Hits running FastAPI server and computes retrieval metrics.
Run server first (uvicorn), then run this script separately.
"""

import os
import time
import random
import requests
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# ---------------- CONFIG ----------------
BASE_URL = "http://localhost:8000"
IMAGES_DIR = r"C:\Users\HP\Downloads\flickr30k_images"
# Flickr30k standard caption file - adjust name if yours differs
CAPTIONS_FILE = "../Images/captions.txt"
NUM_QUERIES = 200          # sample size (keep reasonable, each hit = 1 request)
TOP_K = 10
MAX_WORKERS = 8            # concurrent requests to simulate load

In [6]:
def load_captions(captions_file):
    """
    Expects a CSV/TSV like: image_name,caption   or  image_name#0<TAB>caption
    Returns dict: image_filename -> list of captions
    """
    img_to_caps = {}
    with open(captions_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith("image"):
                continue
            if "\t" in line:
                img_part, caption = line.split("\t", 1)
                img_name = img_part.split("#")[0]
            elif "," in line:
                img_name, caption = line.split(",", 1)
            else:
                continue
            img_to_caps.setdefault(img_name.strip(), []).append(caption.strip())
    return img_to_caps


def build_gt_from_caption(img_name):
    """Ground truth for a text query = the source image filename."""
    return {img_name}

### TEXT -> IMAGE BENCHMARK

In [ ]:
# TEXT -> IMAGE BENCHMARK
def run_text_query(caption, gt_image):
    t0 = time.perf_counter()
    try:
        resp = requests.post(
            f"{BASE_URL}/search/text",
            json={"query": caption, "top_k": TOP_K},
            timeout=15,
        )
        latency_ms = (time.perf_counter() - t0) * 1000
        resp.raise_for_status()
        data = resp.json()
        retrieved = [r["image_url"].split("/")[-1] for r in data.get("results", [])]
    except Exception as e:
        return None
    return {"latency_ms": latency_ms, "retrieved": retrieved, "gt": {gt_image}}


def benchmark_text_to_image(img_to_caps):
    samples = []
    items = list(img_to_caps.items())
    random.shuffle(items)
    for img_name, caps in items[:NUM_QUERIES]:
        caption = random.choice(caps)
        samples.append((caption, img_name))

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(run_text_query, cap, img) for cap, img in samples]
        for fut in as_completed(futures):
            r = fut.result()
            if r:
                results.append(r)
    return compute_metrics(results, "TEXT -> IMAGE")

### IMAGE -> IMAGE BENCHMARK

In [ ]:
# IMAGE -> IMAGE BENCHMARK
def run_image_query(img_path, gt_image):
    t0 = time.perf_counter()
    try:
        with open(img_path, "rb") as f:
            files = {"file": (os.path.basename(img_path), f, "image/jpeg")}
            resp = requests.post(
                f"{BASE_URL}/search/image",
                files=files,
                params={"top_k": TOP_K},
                timeout=20,
            )
        latency_ms = (time.perf_counter() - t0) * 1000
        resp.raise_for_status()
        data = resp.json()
        retrieved = [r["image_url"].split("/")[-1] for r in data.get("results", [])]
    except Exception:
        return None
    return {"latency_ms": latency_ms, "retrieved": retrieved, "gt": {gt_image}}


def benchmark_image_to_image(img_to_caps):
    all_images = list(img_to_caps.keys())
    random.shuffle(all_images)
    sample_images = all_images[:NUM_QUERIES]

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = []
        for img_name in sample_images:
            img_path = os.path.join(IMAGES_DIR, img_name)
            if os.path.exists(img_path):
                # ground truth = itself, since nearest neighbor of an image should be itself
                futures.append(ex.submit(run_image_query, img_path, img_name))
        for fut in as_completed(futures):
            r = fut.result()
            if r:
                results.append(r)
    return compute_metrics(results, "IMAGE -> IMAGE")

### METRICS EVALUATION

In [ ]:
# METRICS
def compute_metrics(results, label, k_values=(1, 5, 10)):
    if not results:
        print(f"[{label}] No successful results.")
        return {}

    latencies = [r["latency_ms"] for r in results]
    recalls = {k: [] for k in k_values}
    reciprocal_ranks = []

    for r in results:
        retrieved, gt = r["retrieved"], r["gt"]
        for k in k_values:
            hit = any(doc in gt for doc in retrieved[:k])
            recalls[k].append(1 if hit else 0)

        rr = 0.0
        for rank, doc in enumerate(retrieved, start=1):
            if doc in gt:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

    metrics = {
        "n_queries": len(results),
        "p50_latency_ms": round(np.percentile(latencies, 50), 2),
        "p95_latency_ms": round(np.percentile(latencies, 95), 2),
        "p99_latency_ms": round(np.percentile(latencies, 99), 2),
        "mean_latency_ms": round(np.mean(latencies), 2),
        "MRR": round(np.mean(reciprocal_ranks), 4),
    }
    for k in k_values:
        metrics[f"recall@{k}"] = round(np.mean(recalls[k]), 4)

    print(f"\n{label} ({metrics['n_queries']} queries) =====")
    print(f"NOTE: It is CPU inference; GPU deployment would reduce this significantly")
    for k, v in metrics.items():
        if k != "n_queries":
            print(f"  {k}: {v}")
    return metrics

### TEST THE EVALUATION

In [16]:
img_to_caps = load_captions(CAPTIONS_FILE)
print(f"Loaded {len(img_to_caps)} images with captions.")
sample_img_to_caps = first_n_dict = dict(list(img_to_caps.items())[:5]) # 5 is sample size
print(f"Loaded {len(sample_img_to_caps)} sample images with captions.")


Loaded 31783 images with captions.
Loaded 5 sample images with captions.


In [19]:
print(benchmark_image_to_image(sample_img_to_caps))


===== IMAGE -> IMAGE (5 queries) =====
  p50_latency_ms: 9776.21
  p95_latency_ms: 10545.73
  p99_latency_ms: 10699.12
  mean_latency_ms: 8622.26
  MRR: 1.0
  recall@1: 1.0
  recall@5: 1.0
  recall@10: 1.0
{'n_queries': 5, 'p50_latency_ms': 9776.21, 'p95_latency_ms': 10545.73, 'p99_latency_ms': 10699.12, 'mean_latency_ms': 8622.26, 'MRR': 1.0, 'recall@1': 1.0, 'recall@5': 1.0, 'recall@10': 1.0}


In [20]:
if __name__ == "__main__":
    print("Loading captions...")
    img_to_caps = load_captions(CAPTIONS_FILE)
    print(f"Loaded {len(img_to_caps)} images with captions.")

    text_metrics = benchmark_text_to_image(img_to_caps)
    image_metrics = benchmark_image_to_image(img_to_caps)

    print("\n===== SUMMARY (copy for your report) =====")
    print("Text->Image:", text_metrics)
    print("Image->Image:", image_metrics)

Loading captions...
Loaded 31783 images with captions.

===== TEXT -> IMAGE (200 queries) =====
  p50_latency_ms: 169.45
  p95_latency_ms: 385.77
  p99_latency_ms: 1122.85
  mean_latency_ms: 221.56
  MRR: 0.9562
  recall@1: 0.92
  recall@5: 1.0
  recall@10: 1.0

===== IMAGE -> IMAGE (200 queries) =====
  p50_latency_ms: 2662.95
  p95_latency_ms: 3714.71
  p99_latency_ms: 4994.9
  mean_latency_ms: 2777.73
  MRR: 1.0
  recall@1: 1.0
  recall@5: 1.0
  recall@10: 1.0

===== SUMMARY (copy for your report) =====
Text->Image: {'n_queries': 200, 'p50_latency_ms': 169.45, 'p95_latency_ms': 385.77, 'p99_latency_ms': 1122.85, 'mean_latency_ms': 221.56, 'MRR': 0.9562, 'recall@1': 0.92, 'recall@5': 1.0, 'recall@10': 1.0}
Image->Image: {'n_queries': 200, 'p50_latency_ms': 2662.95, 'p95_latency_ms': 3714.71, 'p99_latency_ms': 4994.9, 'mean_latency_ms': 2777.73, 'MRR': 1.0, 'recall@1': 1.0, 'recall@5': 1.0, 'recall@10': 1.0}


### PROXY-IMAGES OR TRANSFORMED IMAGES

#### NOTE: Run this first Images\generate_proxy.py then begin

In [25]:
import csv

PROXY_DIR = r"C:\Users\HP\Downloads\flickr30k_proxy_images"
PROXY_CSV = os.path.join(PROXY_DIR, "proxy_ground_truth.csv")


def load_proxy_mapping(csv_path):
    """Returns list of (proxy_filename, ground_truth_filename)."""
    mapping = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            mapping.append((row["proxy_filename"], row["ground_truth_filename"]))
    return mapping


def benchmark_image_to_image_proxy():
    mapping = load_proxy_mapping(PROXY_CSV)
    print(f"Loaded {len(mapping)} proxy queries.")
    print(f"NOTE: It is CPU inference; GPU deployment would reduce this significantly")

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = []
        for proxy_name, gt_name in mapping:
            proxy_path = os.path.join(PROXY_DIR, proxy_name)
            if os.path.exists(proxy_path):
                futures.append(ex.submit(run_image_query, proxy_path, gt_name))
        for fut in as_completed(futures):
            r = fut.result()
            if r:
                results.append(r)

    return compute_metrics(results, "IMAGE -> IMAGE (proxy/robustness)")

if __name__ == "__main__":
    image_metrics = benchmark_image_to_image_proxy()

    print("\nSUMMARY (copy the report) =====")
    print("Image->Image (proxy):", image_metrics)

Loaded 200 proxy queries.
NOTE: It is CPU inference; GPU deployment would reduce this significantly

===== IMAGE -> IMAGE (proxy/robustness) (200 queries) =====
  p50_latency_ms: 2573.12
  p95_latency_ms: 3058.15
  p99_latency_ms: 4718.74
  mean_latency_ms: 2669.36
  MRR: 0.99
  recall@1: 0.985
  recall@5: 0.995
  recall@10: 0.995

SUMMARY (copy the report) =====
Image->Image (proxy): {'n_queries': 200, 'p50_latency_ms': 2573.12, 'p95_latency_ms': 3058.15, 'p99_latency_ms': 4718.74, 'mean_latency_ms': 2669.36, 'MRR': 0.99, 'recall@1': 0.985, 'recall@5': 0.995, 'recall@10': 0.995}
